# LTIC: H22-clean features on Kaggle (Herbarium 2022)

Builds the leakage-free H22-clean split (plan §5.2) and extracts frozen-encoder features in chunks.
Several Kaggle accounts can share the work: account K of N sets `PART = "K/N"`. Full guide: `KAGGLE_H22.md`.

Right-hand panel before running:
1. **Accelerator:** `GPU T4 x2`
2. **Internet:** On (code and encoder weights)
3. **Input:** the competition **Herbarium 2022 - FGVC9** (join it first and accept its rules)

Then **Save Version -> Save & Run All (Commit)**. It keeps running after you close the browser.

In [ ]:
# ---- Settings: the only cell you need to edit ----
GITHUB_USER = "dgpro06"
BRANCH      = "Dgpro"

ENCODERS          = "dinov2_b14,bioclip2,clip_b32"   # one decode pass feeds all three
PART              = "0/3"          # account 1: "0/3", account 2: "1/3", account 3: "2/3"  (one account alone: "0/1")
PREV              = ""             # "auto" = copy every attached earlier H22 output (to continue, or to merge parts)
BASELINES         = ""             # only once ALL chunks are present, e.g. "la,crt,tau_norm,ltic3,crt+c2"
BASELINE_ENCODERS = ""             # default: ENCODERS
TRIALS            = 10

## Step 1: check the GPU and the inputs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input

## Step 2: get the code from GitHub

In [ ]:
import os
CODE = "/kaggle/working/LTIC-Herb"
if os.path.isdir(CODE):
    !git -C {CODE} pull
else:
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/LTIC-Herb.git {CODE}
!git -C {CODE} log --oneline -1

## Step 3: split + features (`kaggle/h22_kaggle.sh`)

In [ ]:
os.environ.update(ENCODERS=ENCODERS, PART=PART, PREV=PREV, BASELINES=BASELINES,
                  BASELINE_ENCODERS=BASELINE_ENCODERS or ENCODERS, TRIALS=str(TRIALS))
!bash {CODE}/kaggle/h22_kaggle.sh
if get_ipython().user_ns.get("_exit_code", 0) != 0:
    raise RuntimeError("The run stopped with an error - scroll up to the first error message.")
!rm -rf {CODE}   # keep the Output small: features, splits, taxonomy and results only